---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-33: LangChain <b>Chains</b> Component</h1>

# Learning agenda of this notebook

> **The idea of this lecture:** real tasks need more than one step: fill a prompt, call a model, clean the answer, then maybe call another model with that answer. A **chain** joins these steps with the `|` operator, so that the output of one step becomes the input of the next, and the whole pipeline runs with a single `invoke()`. This way of writing chains is called **LCEL**, the LangChain Expression Language.

1. **Runnables and Chains**
    - What a Runnalbe?
    - What is a Chain?
    - Setup: check the API keys
2. **Your First Chain**
    - Example 1: `prompt | model | parser`, compared with the same three steps by hand
    - Example 2: A chain is a Runnable too: `batch()` and `stream()`
3. **Output Parsers**
    - Example 3: `PydanticOutputParser`: from the model's text to a Python object
4. **Three Shapes of Chains** (each starts with a warm-up without a model)
    - Example 4: Sequential: translate, then summarise (`RunnableLambda`)
    - Example 5: Parallel: two reports at the same time, then one merged report (`RunnableParallel`)
    - Example 6: Branching (if-else): a different reply for positive, negative and mixed feedback (`RunnableBranch`)
5. **Where Chains Go Next**
6. **Summary**

> **Note:** all examples use the free model `openai/gpt-oss-120b` on Groq.

# <span style='background :lightgreen' >1. Runnables and Chains</span>

## What is a Runnable?
- A **Runnable** is LangChain's common interface for any component that takes an input and produces an output. Its core methods are `invoke()` (one input, one output), `batch()` (many inputs), and `stream()` (output in chunks), plus async variants such as `ainvoke()`. 
- Important classes, at an abstract level:
  - **`Runnable`**: the base interface. Prompt templates, chat models, output parsers, retrievers, and tools all implement it.
  - **`RunnableLambda`**: wraps an ordinary Python function so it can be used as a step.
  - **`RunnableSequence`**: what `a | b | c` builds. It runs the steps in order and feeds each output into the next.
  - **`RunnableParallel`**: runs several Runnables on the same input at once and returns their results together, usually written as a dict inside a chain.
  - **`RunnablePassthrough`**: returns its input unchanged. It is often used to forward the original input alongside computed values.
  - **`RunnableBranch`**: routes the input to one of several Runnables based on conditions.

> **Documentation:** https://reference.langchain.com/python/langchain-core/runnables

## What is a chain?
- In Lec-31 we filled a template with `invoke()` and gave the result to a model with another `invoke()`. Prompt templates, models and output parsers all have the **same** `invoke()` method: in LangChain they are all **Runnables**.
- Because they share one interface, they can be joined with `|`: `prompt | model | parser` is a new Runnable that calls the three in order and passes each output to the next. This is a **chain**.
- A chain is a Runnable again, so it has the same methods (`invoke`, `batch`, `stream`, …) and can itself be a step of a bigger chain.

## Setup: check the API keys
- Every example below is **standalone**: its first code cell loads the API keys and creates the model with `init_chat_model("groq:openai/gpt-oss-120b")` (Lec-30). The later cells of the same example build on it, so run them in order.
- To use OpenAI instead (paid), replace `"groq:openai/gpt-oss-120b"` with `"openai:gpt-4o-mini"` in an example's first cell.
- This cell only checks that the Groq key is found; no example depends on it.

In [1]:
import os                                                   # read the environment variables
from dotenv import load_dotenv                              # load the API keys from ../keys/.env

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ (the file wins)

if os.getenv("GROQ_API_KEY"):                               # the only key the examples need (Groq, in every example)
    print("GROQ_API_KEY set")
else:
    print("GROQ_API_KEY MISSING")                           # never print any part of a key

GROQ_API_KEY set


# <span style='background :lightgreen' >2. Your First Chain</span>

<div style="text-align:center;">
    <img src="../images/langchain-new-16.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## <span style='color:#7e22ce'>Example 1a: <code>prompt | model | parser</code>, compared with the same three steps by hand</span>
### Three steps by hand
We translate a short biography into Urdu. Each step's output is passed **by our code** to the next step.

In [2]:
# Step 1: create the prompt template
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders (Lec-31)

# {language} and {text} are placeholders: invoke() fills them from a dict with the same two keys
translate_prompt = PromptTemplate.from_template("Translate the following English text into {language}. Reply with the translation only.\n\n{text}")
print("Placeholders to fill:", translate_prompt.input_variables)   # ['language', 'text']

Placeholders to fill: ['language', 'text']


In [3]:
# Step 2: fill the template → a PromptValue

# the English text we need to translate ('''...''' lets a string run over several lines)
BIO = '''Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.
He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching
and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a
technology entrepreneur and the founder of Excaliat and Falcon-Hunt.'''

# the two place holders one value for each {placeholder}: the keys must match the names
prompt_value = translate_prompt.invoke({"language": "Urdu", "text": BIO} )        
print(prompt_value)
print()
print()

# PromptValue → a plain str, only to look at it in a human readable way
prompt_text = prompt_value.to_string()                      
print(prompt_text)

text='Translate the following English text into Urdu. Reply with the translation only.\n\nDr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.\nHe holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching\nand management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a\ntechnology entrepreneur and the founder of Excaliat and Falcon-Hunt.'


Translate the following English text into Urdu. Reply with the translation only.

Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.
He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching
and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a
technology entrep

In [4]:
# Step 3: load the API keys, create the model, and invoke the model with prompt_value

from dotenv import load_dotenv                              # reads the API keys from a file
from langchain.chat_models import init_chat_model           # creates a chat model from "provider:model"

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ
model = init_chat_model(
                        "groq:openai/gpt-oss-120b",         # free, on Groq. Yiu can use  "openai:gpt-4o-mini" which is paid on OpenAI
                        max_retries=5,                      # wait and retry when the free rate limit is hit
                        )

# call the model → an AIMessage
ai_message = model.invoke(prompt_value)                     # PromptValue → AIMessage (one call to Groq)

# An AIMessage holds more than the reply text like usage_metadata (the token counts) and response_metadata (the model name, why it stopped). Most of the time we only want the text.
print(ai_message)

content='ڈاکٹر محمد عارف بٹ، یونیورسٹی آف دی پنجاب، لاہور کے ڈیٹا سائنس ڈپارٹمنٹ میں اسسٹنٹ پروفیسر ہیں۔ ان کے پاس کمپیوٹر سائنس میں ایم ایس سی، ایم فل (دونوں گولڈ میڈل کے ساتھ) اور پی ایچ ڈی ہے۔ تینتیس سال سے زائد تدریس اور انتظامیہ کے تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سیکیورٹی اور مصنوعی ذہانت پڑھاتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینر بھی ہیں اور ایکسیلیئٹ اور فیلکن‑ہنٹ کے بانی ہیں۔' additional_kwargs={'reasoning_content': 'The user wants translation of English text into Urdu, reply with translation only. Provide Urdu translation.\n\nNeed to ensure proper Urdu script. Provide translation.\n\nLet\'s translate:\n\n"Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore."\n\nUrdu: "ڈاکٹر محمد عارف بٹ، یونیورسٹی آف دی پنجاب، لاہور کے ڈیٹا سائنس ڈپارٹمنٹ میں اسسٹنٹ پروفیسر ہیں۔"\n\n"He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science."\n\nUrdu: "ان کے پاس کمپیوٹر سائنس میں ایم ایس

In [5]:
# Step 4: keep only the text → a str
translation = ai_message.text                               # AIMessage → str: here OUR code does this step, by hand
print(translation)

ڈاکٹر محمد عارف بٹ، یونیورسٹی آف دی پنجاب، لاہور کے ڈیٹا سائنس ڈپارٹمنٹ میں اسسٹنٹ پروفیسر ہیں۔ ان کے پاس کمپیوٹر سائنس میں ایم ایس سی، ایم فل (دونوں گولڈ میڈل کے ساتھ) اور پی ایچ ڈی ہے۔ تینتیس سال سے زائد تدریس اور انتظامیہ کے تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سیکیورٹی اور مصنوعی ذہانت پڑھاتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینر بھی ہیں اور ایکسیلیئٹ اور فیلکن‑ہنٹ کے بانی ہیں۔


## <span style='color:#7e22ce'>Example 1b: <code>prompt | model | parser</code>, creating a chain and invoking it</span>
- **`StrOutputParser()`** is the simplest output parser: it takes the `AIMessage` from the model and returns only its text, a `str`, the same as `.text` in Step 4 of Example 1a. The metadata (token counts, ids) is dropped, and a plain `str` goes through unchanged.
- Because it is a component, it can be the last step of a chain: `translate_prompt | model | parser`. One `invoke()` runs all three steps.

In [6]:
# Creating  a chain and invoking it (prompt | model | parser)

from dotenv import load_dotenv                                  # reads the API keys from a file
from langchain_core.prompts import PromptTemplate               # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser       # AIMessage -> plain str
from langchain.chat_models import init_chat_model               # creates a chat model from "provider:model"

# Load the API keys into os.environ
load_dotenv("../keys/.env", override=True)

# Create the prompt template, with {language} and {text} placeholders
translate_prompt = PromptTemplate.from_template("Translate the following English text into {language}. Reply with the translation only.\n\n{text}")

# Create the chat model 
model = init_chat_model(
                        "groq:openai/gpt-oss-120b",
                        max_retries=5                                     # wait and retry when the free rate limit is hit
                        )

# You can create two chains
chain1 = translate_prompt | model                      # returns an AIMessage
chain2 = translate_prompt | model | StrOutputParser()  # StrOutputParser() is the simplest output parser that takes the AIMessage from the model and returns only its text


In [7]:
# Invoke the whole chain once by passing it the required place holders for the prompt template and print the translation

# The English text to translate
BIO = '''Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.
He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching
and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a
technology entrepreneur and the founder of Excaliat and Falcon-Hunt.'''

# Invoke chain1, to extract the text  from the response and drop the metadata (token counts, ids), you need to use .text
translation1 = chain1.invoke({"language": "Urdu", "text": BIO})
print(translation1.text)

print()
print()

# chain2 is using StrOutputParser() at the end, so it automatically extracts the text and drop the metadata, and you DONOT need to use .text
translation2 = chain2.invoke({"language": "Urdu", "text": BIO})
print(translation2)

ڈاکٹر محمد عارف بٹ ڈیٹا سائنس کے شعبے، یونیورسٹی آف دی پنجاب، لاہور میں اسسٹنٹ پروفیسر ہیں۔ ان کے پاس کمپیوٹر سائنس میں ایم ایس سی، ایم فل (دونوں سونے کے میڈل کے ساتھ) اور پی ایچ ڈی ہے۔ تعلیم اور انتظامیہ میں 33 سال سے زیادہ کے تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سیکیورٹی اور مصنوعی ذہانت کی تدریس کرتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینیور بھی ہیں اور ایکسیلیئٹ اور فیلکن ہنٹ کے بانی ہیں۔


ڈاکٹر محمد عارف بٹ ڈیپارٹمنٹ آف ڈیٹا سائنس، یونیورسٹی آف دی پنجاب، لاہور میں اسسٹنٹ پروفیسر ہیں۔ ان کے پاس کمپیوٹر سائنس میں ایم ایس سی، ایم فل (دونوں کے ساتھ گولڈ میڈل) اور پی ایچ ڈی ہے۔ تعلیم اور انتظامیہ میں 33 سے زائد سال کے تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سکیورٹی اور مصنوعی ذہانت کی تدریس کرتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینیور بھی ہیں اور ایکسیلیئٹ اور فالفن‑ہنٹ کے بانی ہیں۔


In [8]:
# drawing of the chain: the input, each component, and the output
graph = chain2.get_graph()                         # every step of the chain becomes one box
graph.print_ascii()                                # the stars (*) show where each output goes; needs the grandalf package

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +----------+         
      | ChatGroq |         
      +----------+         
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


## <span style='color:#7e22ce'>Example 2: A chain is a Runnable so you can call <code>stream()</code> and <code>batch()</code></span>
#### `stream(...)` yields the output piece by piece. Because the last step is `StrOutputParser`, the pieces are plain strings.

In [9]:
from dotenv import load_dotenv                                  # reads the API keys from a file
from langchain_core.prompts import PromptTemplate               # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser       # AIMessage -> plain str
from langchain.chat_models import init_chat_model               # creates a chat model from "provider:model"

# Load the API keys into os.environ
load_dotenv("../keys/.env", override=True)

# Create the prompt template, with {language} and {text} placeholders
translate_prompt = PromptTemplate.from_template("Translate the following English text into {language}. Reply with the translation only.\n\n{text}")

# Create the chat model 
model = init_chat_model(
                        "groq:openai/gpt-oss-120b",
                        max_retries=5                                     # wait and retry when the free rate limit is hit
                        )

# Create the chain
translate_chain = translate_prompt | model | StrOutputParser()   # dict → PromptValue → AIMessage → str

# The English text to translate
BIO = '''Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.
He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching
and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a
technology entrepreneur and the founder of Excaliat and Falcon-Hunt.'''

# The model sends small AIMessageChunk pieces; StrOutputParser passes on the text of each piece at once,
# so every piece we get here is a short str
for piece in translate_chain.stream({"language": "Urdu", "text": BIO}):                # each piece is a str (thanks to StrOutputParser)
    print(piece, end="", flush=True)                        # end="": no new line; flush=True: show it at once


ڈاکٹر محمد عارف بٹ ڈیٹا سائنس کے شعبے، یونیورسٹی آف دی پنجاب، لاہور میں اسسٹنٹ پروفیسر ہیں۔ ان کے پاس ایم ایس سی، ایم فل (دونوں کے ساتھ گولڈ میڈل) اور کمپیوٹر سائنس میں پی ایچ ڈی ہے۔ 33 سال سے زائد تدریسی اور انتظامی تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سیکیورٹی اور مصنوعی ذہانت کی تعلیم دیتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینیور بھی ہیں اور ایکسیلیئٹ اور فالکن-ہنٹ کے بانی ہیں۔

### `batch([...])` runs the chain on several inputs at the same time and returns the outputs in the same order.

In [10]:
from dotenv import load_dotenv                                  # reads the API keys from a file
from langchain_core.prompts import PromptTemplate               # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser       # AIMessage -> plain str
from langchain.chat_models import init_chat_model               # creates a chat model from "provider:model"

# Load the API keys into os.environ
load_dotenv("../keys/.env", override=True)

# Create the prompt template, with {language} and {text} placeholders
translate_prompt = PromptTemplate.from_template("Translate the following English text into {language}. Reply with the translation only.\n\n{text}")

# Create the chat model 
model = init_chat_model(
                        "groq:openai/gpt-oss-120b",
                        max_retries=5                                     # wait and retry when the free rate limit is hit
                        )

# Create the chain
translate_chain = translate_prompt | model | StrOutputParser()   # dict → PromptValue → AIMessage → str

# We need to translate the message in three languages
message = "Knowledge is the light that guides us through the darkest times."
languages = ["Urdu", "French", "Arabic"]

# Create a list of three dictionaries to be passed to the .batch()
inputs = []                                                 # one input dict per language
for language in languages:
    inputs.append({"language": language, "text": message})

print(inputs) # printing for your understanding

[{'language': 'Urdu', 'text': 'Knowledge is the light that guides us through the darkest times.'}, {'language': 'French', 'text': 'Knowledge is the light that guides us through the darkest times.'}, {'language': 'Arabic', 'text': 'Knowledge is the light that guides us through the darkest times.'}]


In [11]:
# batch() runs the whole chain once per input dict, the runs at the same time (in threads).

outputs = translate_chain.batch(inputs)             # a list of str, in the same order as inputs
outputs

['علم وہ روشنی ہے جو ہمیں سب سے تاریک وقتوں میں رہنمائی کرتی ہے۔',
 'La connaissance est la lumière qui nous guide à travers les temps les plus sombres.',
 'المعرفة هي النور الذي يوجهنا عبر أحلك الأوقات.']

# <span style='background :lightgreen' >3. Output Parsers</span>

An **output parser** is the last step of many chains: it turns the model's answer into the type our program needs.

| Parser | Turns the answer into | How |
|---|---|---|
| `StrOutputParser()` | a `str` | takes the text of the `AIMessage` |
| `PydanticOutputParser(pydantic_object=Book)` | a validated `Book` object | asks for JSON in the prompt, then reads and checks it |


A model can only send back **text**. To get a Python object with checked fields, `PydanticOutputParser` works in two halves, around the model:

| Method | When | What it does |
|---|---|---|
| `get_format_instructions()` | **before** the model, once | builds, from our Pydantic class, a text that tells the model which JSON to reply with. We put it into the prompt with `partial()` (Lec-31, Example 3). |
| `parse(text)` | **after** the model | reads the JSON in a string (removing any markdown code fences around it), checks every field and its type, and returns a `Book` object. If the JSON is wrong, it raises an `OutputParserException` instead of letting bad data into the program. |
| `invoke(ai_message)` | what the chain calls | takes the text of the `AIMessage` and calls `parse()` on it |

## <span style='color:#7e22ce'>Example 3: <code>PydanticOutputParser</code>: from the model's text to a Python object</span>

In [12]:
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
import rich

load_dotenv("../keys/.env", override=True)                  # load the API key
model = init_chat_model("groq:openai/gpt-oss-120b", temperature=0)         # a free model on Groq

# 1. The answer we want, as a Pydantic class: one field per value we ASK for
#    - the TYPE (str, int) is what the parser checks later: "year" must be a whole number
#    - the DESCRIPTION is copied into the format instructions, so the model reads it
class Book(BaseModel):  # the object we want back
    title: str = Field(description="the book's full title")             # must be a string
    author: str = Field(description="the author's full name")           # must be a string
    year: int = Field(description="the year the book was published")         # must be an integer
    publisher: str = Field(description="the publisher's name")                # must be a string
    isbn: str = Field(description="the book's 13-digit ISBN, digits only")    # a string, not an int: an ISBN is an identifier, not a number to calculate with

# 2. Make a parser for that class
parser = PydanticOutputParser(pydantic_object=Book)

# 3. The parser writes instructions that tell the model which JSON to reply with
# The printed text has three parts:
#   1. "The output should be formatted as a JSON instance ..."  → reply with JSON
#   2. the "foo" example                                        → one right and one wrong JSON, so the model sends
#                                                                 the VALUES, not a copy of the schema
#   3. "Here is the output schema:"                             → our Book class as a JSON schema: each field
#                                                                 with its type and description, and "required" (must be there)
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "the book's full title", "title": "Title", "type": "string"}, "author": {"description": "the author's full name", "title": "Author", "type": "string"}, "year": {"description": "the year the book was published", "title": "Year", "type": "integer"}, "publisher": {"description": "the publisher's name", "title": "Publisher", "type": "string"}, "isbn": {"description": "the book's 13-digit ISBN, digits only", "title": "Isbn", "type": "string"}}, "required": ["title", "author", "year", "publisher", "isbn"]}
``

In [13]:
# 4. # The template has three placeholders: {title}, {author}, and {format_instructions}
# Let us just put the instructions into the prompt using .partial() that pre-fills one of the template's variables and returns a new PromptTemplate. 
# The result still expects title and author later

prompt = PromptTemplate.from_template("Give the publication year, publisher and ISBN of the book '{title}' by {author}.\n{format_instructions}")
prompt = prompt.partial(format_instructions=parser.get_format_instructions())
print(prompt)

input_variables=['author', 'title'] input_types={} partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"title": {"description": "the book\'s full title", "title": "Title", "type": "string"}, "author": {"description": "the author\'s full name", "title": "Author", "type": "string"}, "year": {"description": "the year the book was published", "title": "Year", "type": "integer"}, "publisher": {"description": "the publisher\'s name", "title": "Publisher", "type": "string"}, "isbn": {"description": "the book\'s 13-digit ISBN, digits onl

In [14]:
# 5. The chain: prompt → model → parser (the parser turns the model's JSON text into a Book object)
chain = prompt | model | parser

book = chain.invoke({"title": "Generative Deep Learning (2nd Edition)", "author": "David Foster"})

# 6. The result is a Book object, so we read each value with a dot
print(book)

title='Generative Deep Learning (2nd Edition)' author='David Foster' year=2022 publisher="O'Reilly Media" isbn='9781492078195'


> **`PydanticOutputParser` or `with_structured_output()`?** Both give a validated object. `with_structured_output()` (Lec-30, Example 9) uses the model provider's built-in JSON mode or tool calling, so it is more reliable when the model supports it. `PydanticOutputParser` only needs text, so it works with **any** model, including small local ones, and many RAG tutorials use it.

# <span style='background :lightgreen' >4. Three Shapes of Chains</span>

<div style="text-align:center;">
    <img src="../images/langchain-new-17.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## <span style='color:#7e22ce'>Example 4: Sequential: translate, then summarise (<code>RunnableLambda</code>)</span>
- **`RunnableLambda(function)`** wraps an ordinary Python function so that it becomes a Runnable: a chain step with `invoke()`, `batch()` and `|`.
- **The rules:** the function takes exactly **one** argument (the output of the step before it); what it returns is the input of the step after it; to pass several values onward, return a **dict**.
- **Why we need it here:** `translate_chain` returns a **str**, but `summarize_prompt` needs a **dict** with `language` and `text`. A `RunnableLambda` sits between them as an **adapter**.
- The first cell is a warm-up without a model; the cells after it use `RunnableLambda` in the chain.

In [1]:
# Understand RunnableLambda that turns an ordinary Python function into a chain step
from langchain_core.runnables import RunnableLambda             # wraps a Python function → a Runnable

# Step 1: an ordinary Python function, called the normal way
def count_words(text):
    '''str → int: how many words are in the text.'''
    words = text.split()                                        # split at the spaces → a list of words
    count = len(words)                                          # how many items the list has
    return count                                                # an int
result = count_words("Knowledge is the light")                  # a normal function call, nothing new yet
print("Step 1, plain function:", result)                        # 4

# Step 2: wrap the SAME function in RunnableLambda → now it has invoke(), batch() and |
count_step = RunnableLambda(count_words)                        # pass the function itself: no () after its name
result = count_step.invoke("Knowledge is the light")            # invoke() calls count_words("Knowledge is the light") for us
print("Step 2, invoke():      ", result)                        # 4: the same answer as Step 1
results = count_step.batch(["one two", "one two three"])        # batch() calls count_words once for each item of the list
print("Step 2, batch():       ", results)                       # [2, 3]: one answer per input, in the same order

# Step 3: the function takes ONE argument; to send several values onward, it returns a dict
def make_report(count):
    '''int → dict. Its ONE argument is whatever the step before it returned.'''
    report = {}                                                 # start with an empty dict
    report["words"] = count                                     # the number that came from the step before
    report["long"] = count > 5                                  # True when the text has more than 5 words
    return report                                               # this dict is the input of the next step

report_step = RunnableLambda(make_report)                       # wrap this function too
result = report_step.invoke(4)                                  # test it alone with a number
print("Step 3, alone:         ", result)                        # {'words': 4, 'long': False}

# Step 4: join the two steps with |: the output of count_step becomes the input of report_step
word_chain = count_step | report_step                           # str ──count_step──► int ──report_step──► dict
result = word_chain.invoke("Knowledge is the light that guides us")   # 7 words go in as ONE str
print("Step 4, the chain:     ", result)                        # {'words': 7, 'long': True}

Step 1, plain function: 4
Step 2, invoke():       4
Step 2, batch():        [2, 3]
Step 3, alone:          {'words': 4, 'long': False}
Step 4, the chain:      {'words': 7, 'long': True}


In [3]:
# Step 1: load the API keys, create the model, and create the two chains

from dotenv import load_dotenv                                  # reads the API keys from a file
from langchain_core.prompts import PromptTemplate               # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser       # AIMessage -> plain str
from langchain_core.runnables import RunnableLambda             # turns a Python function into a chain step
from langchain.chat_models import init_chat_model               # creates a chat model from "provider:model"

# Load the API keys into os.environ
load_dotenv("../keys/.env", override=True)

# Create the chat model
model = init_chat_model(
                        "groq:openai/gpt-oss-120b",             # free, on Groq
                        max_retries=5,                          # wait and retry when the free rate limit is hit
                        )

# The English text to translate (the same as in Example 1)
BIO = '''
Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. 
He holds an MSc and MPhil (both with Gold Medals) and a Ph.D. in Computer Science from PUCIT, University of the Punjab. 
His research focuses on fuzzy inference models applied to operating systems, embedded systems, and cloud-based services, 
particularly in decision-making under uncertain and imprecise conditions. With over 33 years of experience in teaching and management, Dr. Butt 
has served in both the Pakistan Army and University of the Punjab, bringing a wealth of interdisciplinary expertise. 
His teaching specializations include embedded and real-time operating systems, system programming, cybersecurity, and artificial intelligence.
Beyond academia, he is a technology entrepreneur, serving as the Founder of Excaliat and Falcon-Hunt and Co-Founder of Tbox Solutionz. 
In recent years, he has gained significant expertise in vulnerability research, binary exploitation, and exploit development, 
excelling in identifying and mitigating critical security risks across diverse platforms. His deep understanding of software architectures, 
memory corruption techniques, and attack vectors strengthens his ability to proactively enhance cybersecurity defences.
A dedicated and results-driven professional, Dr. Butt is recognized for his strong organizational skills, strategic thinking, 
and ability to thrive in collaborative environments. His expertise in cybersecurity and emerging technologies enables him to 
contribute effectively to both academic research and industry innovation, reinforcing defences against evolving cyber threats.
'''
#######################   Chain 1 #########################################
# Chain 1: Given the text it translates to Urdu language:  {language, text} → PromptValue → AIMessage → str (the translation)
translate_prompt = PromptTemplate.from_template("Translate the following English text into {language}. Reply with the translation only.\n\n{text}")
translate_chain = translate_prompt | model | StrOutputParser()
translation = translate_chain.invoke({"language": "Urdu",   "text": BIO})
print(translation)        # StrOutputParser returns a plain str, so this prints the translation
print()
print()
print()
print()
#######################   Chain 2 #########################################
# Chain 2:   Given the translated text it summarizes {language, text} → PromptValue → AIMessage → str (the summary)
summarize_prompt = PromptTemplate.from_template("Summarize the following text in {language} in one sentence. Reply with the summary only.\n\n{text}")
summarize_chain = summarize_prompt | model | StrOutputParser()
summary = summarize_chain.invoke({"language": "Urdu", "text": translation})   # Feed the translation from chain 1 into chain 2
print(summary)

ڈاکٹر محمد عارف بٹ ڈیٹا سائنس کے شعبے، یونیورسٹی آف دی پنجاب (پی یو)، لاہور، پاکستان میں ایک کامیاب اسسٹنٹ پروفیسر ہیں۔ انہوں نے ایم ایس سی اور ایم فل (دونوں میں گولڈ میڈل) کے ساتھ ساتھ کمپیوٹر سائنس میں پی ایچ ڈی بھی پی یو سی آئی ٹی، یونیورسٹی آف دی پنجاب سے حاصل کی ہے۔ ان کی تحقیق فازی استدلال ماڈلز پر مرکوز ہے جو آپریٹنگ سسٹمز، ایمبیڈڈ سسٹمز اور کلاؤڈ بیسڈ سروسز میں لاگو ہوتے ہیں، خاص طور پر غیر یقینی اور غیر واضح حالات میں فیصلہ سازی کے لیے۔ 33 سال سے زیادہ تدریس اور انتظامی تجربے کے ساتھ، ڈاکٹر بٹ نے پاکستان آرمی اور یونیورسٹی آف دی پنجاب دونوں میں خدمات انجام دی ہیں، جس سے انہیں بین الضابطہ مہارت حاصل ہوئی ہے۔ ان کی تدریسی خصوصییتوں میں ایمبیڈڈ اور رئیل ٹائم آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سیکیورٹی اور مصنوعی ذہانت شامل ہیں۔  
اکیڈمیا کے باہر، وہ ایک ٹیکنالوجی انٹرپرینیور ہیں؛ ایکسیلیئٹ اور فالکن-ہنٹ کے بانی اور ٹی باکس سولیوشنز کے شریک بانی ہیں۔ حالیہ سالوں میں انہوں نے خطرے کی تحقیق، بائنری ایکسپلائٹیشن اور ایکسپلائٹ ڈویلپمنٹ میں نمایاں مہارت حاصل کی ہے، مختلف پلیٹ فارمز پ

In [4]:
#######################   Chain 1 #########################################
# Translate the English text into Urdu:  {language, text} → PromptValue → AIMessage → str
translate_prompt = PromptTemplate.from_template("Translate the following English text into {language}. Reply with the translation only.\n\n{text}")
translate_chain = translate_prompt | model | StrOutputParser()

#######################   Chain 2 #########################################
# Summarize the translated text:  {language, text} → PromptValue → AIMessage → str
summarize_prompt = PromptTemplate.from_template("Summarize the following text in {language} in one sentence. Reply with the summary only.\n\n{text}")
summarize_chain = summarize_prompt | model | StrOutputParser()

#######################   Glue step #########################################
# RunnableLambda turns a normal Python function into a chain step.
# Chain 1 gives us a plain string (the translation); chain 2 needs a dict with "language" and "text".
def to_summary_input(translation):
    return {"language": "Urdu", "text": translation}

#######################   One chain, one call #########################################
bio_chain = translate_chain | RunnableLambda(to_summary_input) | summarize_chain

summary = bio_chain.invoke({"language": "Urdu", "text": BIO})
print(summary)

ڈاکٹر محمد عارف بٹ، یونیورسٹی آف دی پنجاب کے اسسٹنٹ پروفیسر، جن کے پاس ایم ایس سی، ایم فل (گولڈ میڈل) اور پی ایچ ڈی ہے، 33 سال سے زائد تدریسی اور انتظامی تجربے کے ساتھ فزی انفراسٹرکچر ماڈلز پر تحقیق، ایمبیڈڈ اور کلاؤڈ سسٹمز میں فیصلہ سازی، سائبر سکیورٹی، بائنری ایکسپلائٹیشن اور سافٹ ویئر آرکیٹیکچر میں مہارت، اور ایکسیلیئٹ، فالکن‑ہنٹ اور ٹی باکس سولیوشنز کے بانی/شریک بانی کے طور پر ٹیکنالوجی انٹرپرینیورشپ کے ساتھ، اپنی اسٹریٹیجک سوچ اور تنظیمی صلاحیتوں سے تعلیمی اور صنعتی دونوں میدانوں میں سائبر دفاع کو مضبوط بناتے ہیں۔


In [7]:
# Step 4: drawing of the chain: make_summary_input is a box of its own, between the two chains
graph = bio_chain.get_graph()                    # every step of the chain becomes one box
graph.print_ascii()                                             # the stars (*) show where each output goes

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +----------+         
      | ChatGroq |         
      +----------+         
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
  +------------------+     
  | to_summary_input |     
  +------------------+     
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

## <span style='color:#7e22ce'>Example 5: Parallel: two reports at the same time, then one merged report (<code>RunnableParallel</code>)</span>
### How the chain in the following code flows:

- **`RunnableParallel({"upper": ..., "length": ...})`** gives the **same** input to every step in the dict and runs them **at the same time**. It returns **one dict** that uses our keys, and each key holds what its step returned: `{"upper": "ARIF BUTT", "length": 9}`.
- Because the steps run together, the call takes about as long as the **slowest** step (≈ 1 s), not the total of both (≈ 2 s).
- **`RunnableLambda(make_sentence)`** wraps a plain Python function so it can be a step in a chain. It calls the function with **one** argument: whatever the previous step returned.
- **`|`** connects the steps. The dict from `parallel` becomes the `data` argument of `make_sentence`, which reads the values using the **same key names** we chose: `data["upper"]`, `data["length"]`.

```
"arif butt" ─┬─► make_upper    ─► "ARIF BUTT" ─► "upper"  ─┐
             └─► count_letters ─► 9           ─► "length" ─┴─► {"upper", "length"} ─► make_sentence ─► str
                                                                                                     "ARIF BUTT has 9 letters"
```

In [15]:
# Understand RunnableParallel that gives ONE input to several steps, and they run AT THE SAME TIME
import time                                                     # sleep(), and a clock to measure the time
from langchain_core.runnables import RunnableLambda             # wraps a Python function → a Runnable
from langchain_core.runnables import RunnableParallel           # runs several Runnables on the same input, at the same time

# Each step sleeps 1 second only to pretend it is slow, like a real model call.
# The timing below shows whether the steps ran one after another (2 seconds) or together (about 1).

def make_upper(text):
    time.sleep(1)                       # pretend this is slow
    return text.upper()                 # "chains" -> "CHAINS"

def count_letters(text):
    time.sleep(1)                       # pretend this is slow
    return len(text)                    # "chains" -> 6

# A dict: each key is a name we choose, and each value is the step that fills it
parallel = RunnableParallel({
                            "upper": RunnableLambda(make_upper),        # the key "upper" will hold what make_upper returns
                            "length": RunnableLambda(count_letters),    # the key "length" will hold what count_letters returns
                            })
# perf_counter() returns the current time in seconds, with very fine precision.
# Reading it before and after the call lets us measure how long the call took.
start = time.perf_counter()             # read the clock before invoke()
result = parallel.invoke("arif butt")   # "arif butt" goes to BOTH steps at once (returns a dict, {'upper': ..., 'length': ...}, not a string.)
seconds = time.perf_counter() - start   # read the clock after invoke() and subtract: the elapsed time

print("Result:", result)                # {'upper': 'ARIF BUTT', 'length': 9}
print(f"Took {seconds:.1f} seconds")    # about 1 second, not 2



# make_sentence() is an ordinary Python function. It takes one argument, a dictionary, and returns a string from two of its keys
def make_sentence(data):
    return f"{data['upper']} has {data['length']} letters"


chain = parallel | RunnableLambda(make_sentence)      # connects the two. The | sends the dict that parallel returns into make_sentence as its data argument.

result = chain.invoke("arif butt")  # What happens on chain.invoke("arif butt")
                                          # Your text goes into parallel. "arif butt" is sent to both of its steps (make_upper, count_letters) at the same time:
                                          # parallel returns a dict. The two results are stored under the keys you chose: {'upper': 'ARIF BUTT', 'length': 9}.
                                          # The | passes that dict to the next step. RunnableLambda receives the dict and calls make_sentence(data) with it.
                                          # make_sentence reads the keys. data['upper'] is 'ARIF BUTT' and data['length'] is 9, so it returns "ARIF BUTT has 9 letters".
print(result)                             # That string becomes the chain's result, and print shows it.

Result: {'upper': 'ARIF BUTT', 'length': 9}
Took 1.0 seconds
ARIF BUTT has 9 letters


In [21]:
# Step 1: load the API keys, create the model, and create one report chain per perspective

from dotenv import load_dotenv                                  # reads the API keys from a file
from langchain_core.prompts import PromptTemplate               # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser       # AIMessage -> plain str
from langchain_core.runnables import RunnableParallel           # runs several chains on the same input, at the same time
from langchain.chat_models import init_chat_model               # creates a chat model from "provider:model"

# Load the API keys into os.environ
load_dotenv("../keys/.env", override=True)

# Create the chat model
model = init_chat_model(
                        "groq:openai/gpt-oss-120b",             # free, on Groq
                        max_retries=5,                          # wait and retry when the free rate limit is hit
                        )

# One report template with two placeholders: {topic} and {perspective}
# "in about 300 words" keeps both reports short enough for the merge step: Groq's free tier rejects
# any single request above 8,000 tokens (HTTP 413), and two long reports together can exceed that
report_prompt = PromptTemplate.from_template("Write a brief analytical report in a single paragraph, in about 100 words, about the {topic} from {perspective}.")

# Two versions of the same template: partial() fixes {perspective}, so each chain needs only {topic}
american_prompt = report_prompt.partial(perspective="the American perspective")
chain_american = american_prompt | model | StrOutputParser()               # {topic} → PromptValue → AIMessage → str (a report)

middle_eastern_prompt = report_prompt.partial(perspective="the Middle Eastern perspective")
chain_middle_eastern = middle_eastern_prompt | model | StrOutputParser()   # {topic} → PromptValue → AIMessage → str (a report)

print("american_prompt needs:      ", american_prompt.input_variables)         # ['topic']: {perspective} is already filled
print("middle_eastern_prompt needs:", middle_eastern_prompt.input_variables)   # ['topic']

american_prompt needs:       ['topic']
middle_eastern_prompt needs: ['topic']


In [22]:
# Step 2: RunnableParallel gives the SAME input to both chains, and returns ONE dict
#   {topic} ─┬─► chain_american       ─► str ─► "report_a"
#            └─► chain_middle_eastern ─► str ─► "report_b"
both_reports = RunnableParallel({
                                "report_a": chain_american,            # key → the chain that fills it (we choose the key names)
                                "report_b": chain_middle_eastern,      # the two chains run in two threads, at the same time
                                })

# ONE input: both chains get this same dict
topic = {"topic": "9/11 incident"}

# invoke() returns when BOTH chains are done (two model calls) that runs in parallel
reports = both_reports.invoke(topic)

# Print the two reports
print("American perspective:\n", reports["report_a"])
print("\nMiddle Eastern perspective:\n", reports["report_b"])

American perspective:
 The September 11, 2001 attacks, orchestrated by al‑Qaida, marked a watershed moment in contemporary American history, reshaping national security policy, foreign relations, and collective consciousness; the coordinated hijackings of four commercial airliners resulted in nearly 3,000 deaths, extensive property loss, and the destruction of the World Trade Center’s Twin Towers, while the crash of United Flight 93 in Pennsylvania averted further catastrophe. In the aftermath, the United States launched the Global War on Terror, authorizing the invasions of Afghanistan and Iraq, expanding the Department of Homeland Security, and enacting the Patriot Act, which provoked vigorous debate over civil liberties versus security imperatives. The event continues to influence American political discourse, immigration attitudes, and cultural narratives of resilience and vigilance.

Middle Eastern perspective:
 From a Middle Eastern viewpoint, the September 11 attacks are often s

In [23]:
# Step 3: the merge chain: its placeholders {report_a} and {report_b} have the SAME names as the keys of Step 2
# (if a key and a placeholder do not match, invoke() raises a KeyError)

merge_prompt = PromptTemplate.from_template("Combine the following two reports into one unified report of fewer than 50 words.\nReport A: {report_a}\nReport B: {report_b}")
merge_chain = merge_prompt | model | StrOutputParser()          # {report_a, report_b} → PromptValue → AIMessage → str

# The dict of Step 2 goes straight in (one model call)
merged = merge_chain.invoke(reports)
print(merged)

The September 11, 2001 al‑Qaida attacks killed nearly 3,000, destroyed the Twin Towers, and launched the U.S. Global War on Terror—Afghanistan/Iraq invasions, DHS, Patriot Act, and civil‑liberty debates. In the Middle East they are viewed as a flashpoint that expanded U.S. military presence, provoked civilian‑casualty resentment and hardened anti‑American sentiment.


In [24]:
# Step 4: the whole pipeline as ONE chain: {topic} → {report_a, report_b} → one short report
final_chain = both_reports | merge_chain                        # a RunnableParallel is a Runnable too, so | works with it

# Two reports at the same time, then the merge (three model calls)
final_report = final_chain.invoke(topic)
print(final_report)

9/11 reshaped U.S. security—prompting the Patriot Act, DHS, and the Global War on Terror—while killing ~3,000. In the Middle East, the attacks are condemned as un‑Islamic, yet U.S. military responses sparked instability, anti‑American sentiment, drone‑strike criticism, and concerns over radicalization.


In [25]:
# Step 5: drawing of the chain: the two report chains side by side, joining again before the merge
graph = final_chain.get_graph()                                 # every step of the chain becomes one box
graph.print_ascii()                                             # the stars (*) show where each output goes

       +----------------------------------+        
       | Parallel<report_a,report_b>Input |        
       +----------------------------------+        
                ***             ***                
              **                   **              
            **                       **            
+----------------+              +----------------+ 
| PromptTemplate |              | PromptTemplate | 
+----------------+              +----------------+ 
          *                             *          
          *                             *          
          *                             *          
    +----------+                  +----------+     
    | ChatGroq |                  | ChatGroq |     
    +----------+                  +----------+     
          *                             *          
          *                             *          
          *                             *          
+-----------------+            +-----------------+ 
| StrOutputP

## <span style='color:#7e22ce'>Example 6: Branching (if-else): a different reply for positive, negative and mixed feedback (<code>RunnableBranch</code>)</span>

### Understanding `RunnableBranch`

In [3]:
# ======================================================================================
# RunnableBranch: an if / elif / else for chain steps
# --------------------------------------------------------------------------------------
# The idea:  check conditions IN ORDER, and run ONLY the step of the FIRST condition
#            that is True. If no condition is True, run the default step.
#
#   5 ─► number_branch ─┬─► is_positive? → reply_positive ─► "5 is positive..."
#                       ├─► is_negative? → reply_negative ─► "-3 is negative..."
#                       └─► (default)    → reply_zero     ─► "0 is zero..."
# ======================================================================================

from langchain_core.runnables import RunnableLambda             # wraps a plain Python function → a Runnable
from langchain_core.runnables import RunnableBranch             # picks ONE step by checking conditions in order


# --- 1. The conditions: they get the input and return True or False -------------------
# A condition only DECIDES which step runs; it does not produce the reply.
def is_positive(number):
    return number > 0                                           # True only for numbers above zero

def is_negative(number):
    return number < 0                                           # True only for numbers below zero

# No is_zero() is needed: zero is what is left when both conditions are False (the default)


# --- 2. The steps: each one produces the reply for ONE case ---------------------------
# The chosen step gets the SAME input that the conditions saw.
def reply_positive(number):
    return f"{number} is positive: it is more than zero."

def reply_negative(number):
    return f"{number} is negative: it is less than zero."

def reply_zero(number):
    return f"{number} is zero: neither positive nor negative."


# --- 3. RunnableBranch: pair each condition with its step, end with a default ---------
# Pattern:  RunnableBranch( (condition, step), (condition, step), ..., default_step )
#
# The same thing in plain Python:
#   if   is_positive(number):  return reply_positive(number)
#   elif is_negative(number):  return reply_negative(number)
#   else:                      return reply_zero(number)
number_branch = RunnableBranch(
                                (is_positive, RunnableLambda(reply_positive)),              # if:   checked FIRST
                                (is_negative, RunnableLambda(reply_negative)),              # elif: checked ONLY when the first was False
                                RunnableLambda(reply_zero),                                 # else: the default, NO condition, always last
                            )


# --- 4. Run it: each number takes its OWN branch --------------------------------------
for number in [5, -3, 0]:                                       # one number for each branch
    reply = number_branch.invoke(number)                        # the input is just the number, no dict needed
    print(reply)

# Output:
#   5 is positive: it is more than zero.                        ← is_positive was True
#   -3 is negative: it is less than zero.                       ← is_positive False, is_negative True
#   0 is zero: neither positive nor negative.                   ← both False, so the default ran

5 is positive: it is more than zero.
-3 is negative: it is less than zero.
0 is zero: neither positive nor negative.


### Complete Example

In [1]:
# ======================================================================================
# RunnableBranch: classify each feedback, then run ONE reply chain chosen by its sentiment
# --------------------------------------------------------------------------------------
# For each feedback:
#   1. classify    "The laptop arrived broken..."  → "negative"        (model call 1)
#   2. build dict  {"feedback": "...", "sentiment": "negative"}        (plain Python)
#   3. branch      checks the sentiment → runs ONLY negative_chain     (model call 2)
#
#   {feedback, sentiment} ─► branch ─┬─► is_positive? → positive_chain ─► reply
#                                    ├─► is_negative? → negative_chain ─► reply
#                                    └─► (default)    → mixed_chain    ─► reply
# ======================================================================================

from dotenv import load_dotenv                                  # reads the API keys from a .env file
from langchain_core.prompts import PromptTemplate               # a text template with named {placeholders}
from langchain_core.output_parsers import StrOutputParser       # AIMessage → plain str
from langchain_core.runnables import RunnableBranch             # if / elif / else: picks ONE chain to run
from langchain.chat_models import init_chat_model               # creates a chat model from "provider:model"

load_dotenv("../keys/.env", override=True)                      # puts the API keys into os.environ
model = init_chat_model("groq:openai/gpt-oss-120b", max_retries=5)   # free model on Groq; retry on rate limits


# --- 1. The classifier: feedback → ONE word ------------------------------------------
# The prompt asks for exactly one word, so the conditions below can compare it with ==
classify = PromptTemplate.from_template("Answer with only one word: positive, negative or mixed.\n" "What is the sentiment of this customer feedback: {feedback}") | model | StrOutputParser()       


# --- 2. Three reply chains: same structure, only the instruction differs --------------
#   {feedback} ─► prompt ─► model ─► StrOutputParser ─► str (a reply)
positive_chain = PromptTemplate.from_template("Thank the customer warmly, in two sentences, for this feedback: {feedback}") | model | StrOutputParser()

negative_chain = PromptTemplate.from_template("Apologise and say the issue goes to the support team, in two sentences, for this feedback: {feedback}") | model | StrOutputParser()

mixed_chain = PromptTemplate.from_template("Thank the customer for the good points and apologise for the problem, in two sentences, for this feedback: {feedback}") | model | StrOutputParser()


# --- 3. The conditions: get the WHOLE dict, return True or False ----------------------
def is_positive(data):
    return data["sentiment"] == "positive"

def is_negative(data):
    return data["sentiment"] == "negative"
# No is_mixed() needed: "mixed" is what is left when both are False (the default)


# --- 4. RunnableBranch: the same as if / elif / else ----------------------------------
#   if   is_positive(data):  positive_chain.invoke(data)
#   elif is_negative(data):  negative_chain.invoke(data)
#   else:                    mixed_chain.invoke(data)
branch = RunnableBranch(
                        (is_positive, positive_chain),                              # if:   checked FIRST
                        (is_negative, negative_chain),                              # elif: checked only if the first was False
                        mixed_chain,                                                # else: the default, NO condition, always last
                        )


# --- 5. Run it on three feedbacks ------------------------------------------------------
feedbacks = [
    "Excellent service, my order arrived a day early!",              # expected → positive
    "The laptop arrived broken and nobody answers my emails.",       # expected → negative
    "The delivery was late but the product quality is excellent.",   # expected → mixed
]

for feedback in feedbacks:
    sentiment = classify.invoke({"feedback": feedback}).strip().lower()   # "Negative\n" → "negative"
    data = {"feedback": feedback, "sentiment": sentiment}       # the branch needs BOTH keys:
                                                                #   "sentiment" → the conditions choose a chain
                                                                #   "feedback"  → the chosen chain fills {feedback}
    reply = branch.invoke(data)                                 # ONLY the chosen chain runs (one model call)

    print(f"Feedback:  {feedback}")
    print(f"Sentiment: {sentiment}")
    print(f"Reply:     {reply}")
    print()

Feedback:  Excellent service, my order arrived a day early!
Sentiment: positive
Reply:     Thank you so much for sharing your wonderful feedback! We're thrilled that our service could get your order to you a day early and exceed your expectations.

Feedback:  The laptop arrived broken and nobody answers my emails.
Sentiment: negative
Reply:     We’re very sorry to hear that your laptop arrived damaged and that you haven’t received a response to your emails. Your case has been forwarded to our support team, and they’ll reach out to you as soon as possible.

Feedback:  The delivery was late but the product quality is excellent.
Sentiment: mixed
Reply:     Thank you for highlighting the excellent quality of our product; we’re delighted you’re pleased with it. We apologize for the delayed delivery and appreciate your patience.



# <span style='background :lightgreen' >5. Where Chains Go Next</span>

| Where | What the chain does there | Lecture |
|---|---|---|
| **RAG** | `retriever` → `prompt` → `model` → `parser`: answers questions from your own documents, with `RunnableParallel` and `RunnablePassthrough` | Lec-38 |
| **Agents** | An agent is not a fixed chain: the model decides the next step in a loop. We write that loop ourselves, then use `create_agent()` | Lec-39 to 51 |
| **LangGraph** | Workflows with loops, saved state, branches between agents and human approval | A later lecture |

# <span style='background :lightgreen' >6. Summary</span>

| Without chains (Lec-31) | With chains (LCEL) |
|---|---|
| `prompt.invoke()`, then `model.invoke()`, then `.text`, passed along by our code | `prompt \| model \| StrOutputParser()`, one `invoke()` |
| A loop of calls for many inputs | `chain.batch([...])` |
| Asking for JSON and parsing it ourselves | `PydanticOutputParser` as the last step |
| Reshaping data between steps by hand | `RunnableLambda(function)` |
| Running steps one after the other | `RunnableParallel({...})` |
| An `if` around separate calls | `RunnablePassthrough.assign(...)` + `RunnableBranch(...)` |

### Key takeaways
- Prompts, models, parsers and functions are all **Runnables**: they share `invoke`, `batch` and `stream`, so they can be joined with `|`.
- A **chain** passes each step's output to the next step, and is itself a Runnable.
- **Output parsers** turn the model's answer into a `str` (`StrOutputParser`) or a validated object (`PydanticOutputParser`).
- Three shapes cover most pipelines: **sequential** (`|`, with `RunnableLambda` to reshape data), **parallel** (`RunnableParallel`), and **branching** (`RunnableBranch`, with `.assign()` to keep the input).
- The old `LLMChain`-style classes are deprecated; loops and state belong to LangGraph.
- **Next lecture (Lec-34):** **Document loaders**: getting your own documents into LangChain.

### Check your understanding
1. What do prompt templates, models and output parsers have in common that lets them be joined with `|`?
2. In Example 1b, which step of Example 1a does `StrOutputParser()` replace?
3. What does `translate_chain.batch(inputs)` return, and in which order?
4. What are the two halves of `PydanticOutputParser`'s work, before and after the model? What does `get_format_instructions()` return, and what happens in `parse()` if the model's JSON is wrong?
5. Why could `translate_chain` and `summarize_chain` not be joined directly in Example 4? What did `RunnableLambda` do, and how many arguments may its function take?
6. What does `RunnableParallel` return, and why does that fit `merge_prompt`? In the warm-up of Example 5, why did two steps of 1 second take about 1 second? When does running in parallel save time with a real model, and when not?
7. In Example 6, what would the reply prompt receive without `RunnablePassthrough.assign()`?
8. How does `RunnableBranch` choose a branch, and what happens when no condition is true?
9. Why does LangChain recommend LangGraph instead of chains for workflows with loops?